# NB10 - Private Inference
## Background
Private inference allows a model owner to perform inference on a client's data without learning the data, and without revealing the model weights. The two main cryptographic approaches are Homomorphic Encryption (HE) and Secure Multi-Party Computation (SMPC).

CKKS (Cheon-Kim-Kim-Song, 2017) is the leading HE scheme for ML: it supports approximate arithmetic over real numbers, enabling encrypted evaluation of polynomials. Since ReLU is non-polynomial, HE-based inference typically replaces activations with polynomial approximations (degree-3 or degree-7 polynomials). SMPC-based methods (CrypTen, MOTION) use secret sharing and avoid the polynomial approximation issue but require multiple rounds of communication.

## What You'll Learn
- CKKS-style encrypted arithmetic simulation
- Polynomial approximation of ReLU for HE compatibility
- Private inference protocol: client encrypts input, server computes, client decrypts
- Accuracy-privacy tradeoff from polynomial activation approximation

## Why This Matters
Private inference is deployed in healthcare ML: a hospital can run a diagnostic model on patient data without the model provider seeing patient records, and without the hospital learning the model weights.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# ── Simulated CKKS-style encrypted arithmetic ──────────────────────────────
class EncryptedFloat:
    """Simulates CKKS encryption with additive noise (precision simulation)."""
    def __init__(self, value: float, noise_scale: float = 1e-6):
        self.value = value
        self._noise = np.random.randn() * noise_scale

    def decrypt(self) -> float:
        return self.value + self._noise

    def __add__(self, other):
        if isinstance(other, EncryptedFloat):
            result = EncryptedFloat(self.value + other.value)
            result._noise = self._noise + other._noise
            return result
        return EncryptedFloat(self.value + other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            # Multiplication by plaintext: noise grows
            result = EncryptedFloat(self.value * other)
            result._noise = self._noise * abs(other) + np.random.randn() * 1e-5
            return result
        return NotImplemented

    def __repr__(self):
        return f'Enc({self.value:.4f})'

# ── Polynomial approximation of ReLU ──────────────────────────────────────
def relu_poly_degree2(x: float) -> float:
    """Degree-2 polynomial approximation of ReLU on [-4, 4]."""
    # Approximation: 0.25*x^2 + 0.5*x + 0.25 (minimax on [0, 4])
    return max(0, 0.25*x**2 + 0.5*x)

def relu_poly_degree3(x: float) -> float:
    """Degree-3 polynomial (better fit)."""
    # Cubic Hermite approximation
    return 0.5*x + 0.375*x**2 - 0.0625*x**3

def relu_poly_degree7(x: float, scale=4.0) -> float:
    """Degree-7 Chebyshev approximation of ReLU."""
    t = x / scale
    t = max(-1, min(1, t))
    cheb = (
        0.5 + 0.5*t +
        -0.0625*t**3 +
        0.0234375*t**5
    )
    return max(0, cheb * scale)

# ── Visualize approximation quality ───────────────────────────────────────
x = np.linspace(-4, 4, 500)
relu_true = np.maximum(0, x)
poly2 = np.array([relu_poly_degree2(xi) for xi in x])
poly3 = np.array([relu_poly_degree3(xi) for xi in x])
poly7 = np.array([relu_poly_degree7(xi) for xi in x])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, relu_true, 'k-', linewidth=2.5, label='True ReLU')
ax.plot(x, poly2, 'r--', linewidth=1.5, label='Degree-2 poly')
ax.plot(x, poly3, 'b--', linewidth=1.5, label='Degree-3 poly')
ax.plot(x, poly7, 'g--', linewidth=1.5, label='Degree-7 poly')
ax.set_xlim(-4, 4); ax.set_ylim(-1, 4)
ax.set_title('Polynomial Approximations of ReLU for HE Inference')
ax.set_xlabel('Pre-activation z'); ax.set_ylabel('Activation output')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s16_10_poly_relu.png', dpi=100, bbox_inches='tight')
plt.show()

# Approximation errors
for name, poly in [('Degree-2', poly2), ('Degree-3', poly3), ('Degree-7', poly7)]:
    mse = np.mean((poly - relu_true)**2)
    print(f'{name}: MSE = {mse:.6f}')

In [ ]:
# ── Private inference simulation ───────────────────────────────────────────
class PrivateInferenceSimulator:
    """Simulates client-server private inference."""

    def __init__(self, W1, b1, W2, b2, poly_degree=3):
        self.W1, self.b1, self.W2, self.b2 = W1, b1, W2, b2
        self.poly_degree = poly_degree

    def poly_relu(self, x: float) -> float:
        if self.poly_degree == 2:
            return relu_poly_degree2(x)
        elif self.poly_degree == 7:
            return relu_poly_degree7(x)
        return relu_poly_degree3(x)  # default: degree 3

    def client_encrypt(self, x: np.ndarray) -> List[EncryptedFloat]:
        return [EncryptedFloat(xi) for xi in x]

    def server_compute(self, enc_x: List[EncryptedFloat]) -> List[EncryptedFloat]:
        # Server computes on encrypted data
        # Layer 1: linear (plaintext weights x encrypted input)
        enc_h1 = []
        for j in range(self.W1.shape[1]):
            val = sum(enc_x[i] * float(self.W1[i, j])
                      for i in range(len(enc_x)))
            val = val + float(self.b1[j])
            enc_h1.append(val)
        # Poly ReLU (server must decrypt-apply-reencrypt in real SMPC,
        # or use poly approx for HE)
        enc_relu = [EncryptedFloat(self.poly_relu(e.decrypt())) for e in enc_h1]
        # Layer 2
        enc_out = []
        for j in range(self.W2.shape[1]):
            val = sum(enc_relu[i] * float(self.W2[i, j])
                      for i in range(len(enc_relu)))
            val = val + float(self.b2[j])
            enc_out.append(val)
        return enc_out

    def client_decrypt(self, enc_out: List[EncryptedFloat]) -> np.ndarray:
        return np.array([e.decrypt() for e in enc_out])

# Generate small model and test
np.random.seed(42)
n_feat, n_h, n_cls = 4, 8, 3
W1 = np.random.randn(n_feat, n_h) * 0.4
b1 = np.zeros(n_h)
W2 = np.random.randn(n_h, n_cls) * 0.4
b2 = np.zeros(n_cls)

# Standard inference
x = np.random.randn(n_feat)
h_std = np.maximum(0, x @ W1 + b1)
out_std = h_std @ W2 + b2

# Private inference
results = {}
for degree in [2, 3, 7]:
    pi = PrivateInferenceSimulator(W1, b1, W2, b2, poly_degree=degree)
    enc_x = pi.client_encrypt(x)
    enc_out = pi.server_compute(enc_x)
    out_priv = pi.client_decrypt(enc_out)
    error = np.abs(out_priv - out_std).max()
    results[degree] = (out_priv, error)
    print(f'Degree-{degree} poly ReLU: max_error={error:.6f}')

print()
print(f'Standard output:         {out_std}')
for degree, (out_priv, err) in results.items():
    print(f'Private (degree-{degree}): {out_priv}')